In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report

df = pd.read_csv('dataset.csv')
df = df[df.track_genre.isin(['turkish','study','rock','rock-n-roll','metal','brazil','folk','opera'])]
df.drop(columns=["track_id","artists","album_name","track_name","n"],inplace=True)
df.explicit= np.where(df.explicit == False,0,1)
df.reset_index(drop=True,inplace=True)
(df.head(15))

,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,82,157444,0,0.848,0.821,2,-5.408,0,0.0527,0.01690,0.000403,0.0962,0.249,125.051,4,brazil
1,56,194478,0,0.833,0.517,11,-7.502,0,0.3010,0.11600,0.000142,0.1160,0.187,129.047,4,brazil
2,54,221775,0,0.720,0.781,10,-2.647,0,0.0609,0.00408,0.807000,0.3080,0.307,125.005,4,brazil
3,57,202213,0,0.692,0.427,11,-10.733,0,0.3910,0.52700,0.000000,0.1700,0.542,75.086,4,brazil
4,54,208000,1,0.704,0.616,6,-7.042,0,0.0818,0.29900,0.000000,0.0856,0.161,89.974,4,brazil
5,52,201459,0,0.750,0.507,11,-9.219,0,0.0580,0.79800,0.000000,0.3100,0.897,136.937,4,brazil
6,52,249490,0,0.799,0.584,2,-8.157,1,0.1180,0.10300,0.000001,0.4570,0.404,117.031,4,brazil
7,55,224973,0,0.632,0.866,1,-5.445,0,0.0488,0.05560,0.000006,0.5180,0.835,87.785,4,brazil
8,53,238000,1,0.611,0.547,7,-7.861,1,0.4390,0.07170,0.000000,0.1570,0.355,158.852,4,brazil
9,53,239072,0,0.641,0.617,5,-7.411,0,0.0307,0.41300,0.000000,0.1120,0.941,139.942,4,brazil


In [7]:
features=df.iloc[:,:-1]
target=df.iloc[:,-1]
x_train,x_test,y_train,y_test=train_test_split(features,target,random_state=12)

In [10]:
n=9
knn=KNeighborsClassifier(n_neighbors=n).fit(x_train,y_train)
knn_accuracy=knn.score(x_test,y_test)
print(f"KNN accuracy is: {knn_accuracy:.2f}%")
y_pred=knn.predict(x_test)
print(classification_report(y_test,y_pred))

KNN accuracy is: 0.30%
              precision    recall  f1-score   support

      brazil       0.21      0.32      0.25       232
        folk       0.24      0.31      0.27       236
       metal       0.25      0.23      0.24       255
       opera       0.26      0.18      0.21       247
        rock       0.32      0.35      0.33       251
 rock-n-roll       0.38      0.39      0.38       260
       study       0.51      0.53      0.52       250
     turkish       0.18      0.09      0.12       269

    accuracy                           0.30      2000
   macro avg       0.29      0.30      0.29      2000
weighted avg       0.29      0.30      0.29      2000



In [11]:
rfc=RandomForestClassifier(random_state=42)
rfc.fit(x_train,y_train)
rfc_accuracy=rfc.score(x_test,y_test)*100
print(f"RFC accuracy is: {rfc_accuracy:.2f}%")
y_pred= rfc.predict(x_test)
print(classification_report(y_test,y_pred))

RFC accuracy is: 78.35%
              precision    recall  f1-score   support

      brazil       0.73      0.80      0.77       232
        folk       0.65      0.53      0.58       236
       metal       0.70      0.75      0.72       255
       opera       0.90      0.89      0.90       247
        rock       0.69      0.71      0.70       251
 rock-n-roll       0.84      0.84      0.84       260
       study       0.94      0.98      0.96       250
     turkish       0.79      0.76      0.77       269

    accuracy                           0.78      2000
   macro avg       0.78      0.78      0.78      2000
weighted avg       0.78      0.78      0.78      2000



In [ ]:
param_grid = {
    'n_estimators': [10,15,50, 100,200,300],
    'max_depth': [1,3, 5,7,10,15,20,None],
    'max_features': ["sqrt", "log2", None],
    'min_samples_leaf': [1, 2, 3,4],
    'bootstrap': [True, False]
}
rfc_tuning=RandomForestClassifier(random_state=42)
from sklearn.model_selection import GridSearchCV
grid_search=GridSearchCV(rfc_tuning,param_grid=param_grid,cv=10,scoring='accuracy',n_jobs=-1,verbose=2)
grid_search.fit(x_train,y_train)

print(f'Best Parameter Combo: {grid_search.best_params_}')
#!Best Parameter Combo: {'bootstrap': False, 'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 300}
print(f'Best Accuracy Score%: {grid_search.best_score_*100}')
#!Best Accuracy Score%: 79.10000000000001

#SINCE IT TAKES 35 MINS, I DIDNT RUN IN MAKING THIS JUPYTER FILE BUT IN MY ORIGINAL ONE, I DID AND SHOWED IN #

Fitting 10 folds for each of 1152 candidates, totalling 11520 fits


KeyboardInterrupt: 

In [15]:
print(
    f"KNN (k=9): 30%\n"
    f"Random Forest: 78.1%\n"
    f"RFC with GridSearchCV: 79.1%"
)

KNN (k=9): 30%
Random Forest: 78.1%
RFC with GridSearchCV: 79.1%
